<a href="https://colab.research.google.com/github/inhyuk78/Re-implementation-of-MOLI-By-Hossein-Sharifi-Noghabi-/blob/main/02_exploratory_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install pyreadr

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import matplotlib.pyplot as plt

In [ ]:
drug_df = pd.read_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Dataframes/drug.parquet')
rnaseq_df = pd.read_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Dataframes/rnaseq.parquet')
mutation_df = pd.read_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Dataframes/mutation.parquet')
CNA_df = pd.read_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Dataframes/CNA.parquet')

In [ ]:
def perform_pca(X, n_components=3):
  '''
  Perform Principal Component Analysis (PCA) on input data.

  Args:
    X (array-like of shape :(n_samples, n_features)): Input data to fit and transform.
    n_components (int, default=3): Number of principal components to keep.

  Returns:
    pca (sklearn.decomposition.PCA): Fitted PCA object.
    variance_ratio (ndarray of shape (n_components, )): Percentage of variance explained by each of the selected components.
    X_pca (ndarray of shape (n_samples, n_components)): Input data transformed into the principal component space.
  '''
  pca = PCA(n_components=n_components)
  pca.fit(X)

  variance_ratio = pca.explained_variance_ratio_
  X_pca = pca.transform(X)
  return pca, variance_ratio, X_pca

## Basic Exploratory Analysis

In [ ]:
print(f'drug_df shape: {drug_df.shape}')
print(f'rnaseq_df shape: {rnaseq_df.shape}')
print(f'mutation_df shape: {mutation_df.shape}')
print(f'CNA_df shape: {CNA_df.shape}')

drug_df shape: (242036, 7)
rnaseq_df shape: (36257567, 4)
mutation_df shape: (4517891, 5)
CNA_df shape: (18667013, 8)


## Cell-line Cancer Type Lookup Dictionary

In [ ]:
drug_df.head()

cancer_map = dict(
    zip(drug_df['SANGER_MODEL_ID'],
        drug_df['CANCER_TYPE'])
)

## Gene Expression Data Exploration

In [ ]:
rnaseq_df.head()

,model_name,model_id,gene_symbol,rsem_tpm
0,LS-123,SIDM00776,DVL2,3.5521
1,MOLM-13,SIDM00437,STPG1,2.2898
2,HARA,SIDM00598,STPG1,2.9486
3,HCC-366,SIDM01070,STPG1,2.4906
4,MV-4-11,SIDM00657,STPG1,1.3674


In [ ]:
dups = rnaseq_df[rnaseq_df.duplicated(subset=['model_id', 'gene_symbol'], keep=False)]

dups.sort_values(['model_id', 'gene_symbol']).head(5)

,model_name,model_id,gene_symbol,rsem_tpm
22427874,M14,SIDM00003,VARS1,6.1568
22427875,M14,SIDM00003,VARS1,6.1568
33190159,TE-12,SIDM00023,VARS1,5.0596
33190161,TE-12,SIDM00023,VARS1,5.0596
24970315,TMK-1,SIDM00040,VARS1,6.6738


In [ ]:
rnaseq_pivot = rnaseq_df.pivot_table(
    index='model_id',
    columns='gene_symbol',
    values='rsem_tpm',
    aggfunc='mean'
)

rnaseq_pivot.head()

gene_symbol,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2ML1-AS1,A2ML1-AS2,A2MP1,A3GALT2,...,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11AP1,ZYG11B,ZYX,ZYXP1,ZZEF1,ZZZ3
model_id,,,,,,,,,,,,,,,,,,,,,
SIDM00003,3.9736,3.1408,0.0144,4.1326,0.4114,0.0000,0.0,0.0,0.0000,0.0841,...,1.0909,2.1010,4.0488,2.6369,0.0976,3.7148,6.9590,0.0,4.3561,5.5622
SIDM00023,0.1110,0.3785,0.2388,0.0286,0.1375,0.0144,0.0,0.0,0.0286,0.0841,...,1.5705,2.6323,4.0108,2.5435,0.0000,3.9439,5.7241,0.0,4.2525,4.5040
SIDM00040,0.0704,0.2016,0.0000,0.2016,0.0841,0.0000,0.0,0.0,0.0000,0.0000,...,0.7655,2.1538,2.4303,2.3813,0.0000,2.7677,5.7477,0.0,2.8718,4.9467
SIDM00041,4.8017,2.9126,0.0286,0.2141,1.7866,0.0286,0.0,0.0,0.0144,0.1243,...,1.2265,2.0426,4.1152,2.4436,0.0566,4.0687,8.4034,0.0,3.9964,5.3740
SIDM00042,0.5059,0.8639,0.0566,0.2265,1.3561,0.4542,0.0,0.0,0.0286,0.0704,...,1.2630,2.5008,4.7840,0.0704,0.0000,2.3813,5.0112,0.0,4.1465,4.2615


In [ ]:
print(f'Shape before data imputation: {rnaseq_pivot.shape}')
print(f'Number of NaN values: {rnaseq_pivot.isna().sum().sum()}')

# Drop columns with >20% missing values (mean imputation on sparse columns invalidates them)
threshold = 0.8 * len(rnaseq_pivot)
rnaseq_pivot = rnaseq_pivot.dropna(axis=1, thresh=int(threshold))

# Mean-impute remaining NaNs
rnaseq_pivot = rnaseq_pivot.fillna(rnaseq_pivot.mean())

print(f'Shape after data imputation: {rnaseq_pivot.shape}')
print(f'Number of NaN values: {rnaseq_pivot.isna().sum().sum()}')

Shape before data imputation: (944, 41143)
Number of NaN values: 2582369
Shape after data imputation: (944, 36416)
Number of NaN values: 0


In [ ]:
scaler = StandardScaler()
rnaseq_scaled = scaler.fit_transform(rnaseq_pivot)
pca, variance_ratio, rnaseq_pca = perform_pca(rnaseq_scaled, n_components=3)

def plot_scatter_3d(X_pca, rnaseq_df, cancer_map, name):
  '''
  Creates an interactive 3D scatter plot of PCA-transformed data coloured by cancer type.

  Args:
    X_pca (ndarray of shape (n_samples, 3)): PCA-transformed data with 3 components (PC1, PC2, PC3).
    rnaseq_df (pandas.DataFrame): Original gene expression data containing 'model_id' column used to align sample IDs with rows of X_pca.
    cancer_map (dict): Mapping from 'SANGER_MODEL_ID' to 'CANCER_TYPE' used to colour/label points.
    name (str): Label used in plot title.

  Returns:
    fig (plotly.graph_objects.Figure): Interactive 3D scatter plot of 3 principal components, coloured by cancer type, with sample ID and cancer type shown on hover.
  '''
  df = pd.DataFrame(X_pca, columns=['PC1', 'PC2', 'PC3'])
  df['SANGER_MODEL_ID'] = rnaseq_df['model_id']
  df['CANCER_TYPE'] = df['SANGER_MODEL_ID'].map(cancer_map)

  fig = px.scatter_3d(
      df,
      x='PC1',
      y='PC2',
      z='PC3',
      color='CANCER_TYPE',
      hover_data=['SANGER_MODEL_ID', 'CANCER_TYPE'],
      title=f'3D PCA plot of {name}'
  )

  fig.update_traces(marker=dict(size=2))
  return fig

fig = plot_scatter_3d(rnaseq_pca, rnaseq_df, cancer_map, 'Gene Expression Data')
fig.show()

## Mutation Data Exploration

In [ ]:
mutation_df.head()

,model_name,model_id,gene_symbol,cancer_driver,effect
0,GR-ST,SIDM01259,TEKT4,0,ess_splice
1,GR-ST,SIDM01259,EIF5B,0,frameshift
2,GR-ST,SIDM01259,NPHP1,0,intronic
3,GR-ST,SIDM01259,NPHP1,0,frameshift
4,GR-ST,SIDM01259,MIR1302-3,0,nc_variant


In [ ]:
dups = mutation_df[mutation_df.duplicated(subset=['model_id', 'gene_symbol'], keep=False)]

dups.sort_values(['model_id', 'gene_symbol']).head(5)

,model_name,model_id,gene_symbol,cancer_driver,effect
3691569,M14,SIDM00003,A2ML1,0,intronic
3691570,M14,SIDM00003,A2ML1,0,intronic
3691571,M14,SIDM00003,A2ML1,0,intronic
3691828,M14,SIDM00003,AACS,0,missense
3691829,M14,SIDM00003,AACS,0,downstream


In [ ]:
mutation_pivot = mutation_df.pivot_table(
    index='model_id',
    columns='gene_symbol',
    values='cancer_driver',
    aggfunc='max'   # aggfunc='max' because cancer_driver is binary: if any duplicate model_id/gene_symbol row has a driver mutation (1), we keep 1
)

mutation_pivot.head()

gene_symbol,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
model_id,,,,,,,,,,,,,,,,,,,,,
SIDM00003,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
SIDM00023,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
SIDM00040,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
SIDM00041,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SIDM00042,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(f'Shape before data imputation: {mutation_pivot.shape}')
print(f'Number of NaN values: {mutation_pivot.isna().sum().sum()}')

mutation_pivot = mutation_pivot.fillna(0)

  # NaN = no mutation record for this model/gene pair, treated as 'not a driver mutation' -> fill with 0
  # (assumes absence of a driver-mutation record is equivalent to a confirmed non-driver call)

print(f'Shape after data imputation: {mutation_pivot.shape}')
print(f'Number of NaN values: {mutation_pivot.isna().sum().sum()}')

Shape before data imputation: (965, 20468)
Number of NaN values: 16946999
Shape after data imputation: (965, 20468)
Number of NaN values: 0


In [ ]:
def plot_oncoheatmap_binary(df, name, color_label):
  '''
  Plots a binary oncoprint-style heatmap of mutation presence/absence.

  Args:
    df (pandas.DataFrame): Data of shape (cell_lines, genes), typically binary (0/1) values.
    name (str): Label used to plot title.
    color_label (str): Label used for colour bar.

  Returns:
    fig (plotly.graph_objects.Figure): Heatmap with cell lines on x-axis, genes on y-axis.
  '''
  fig = px.imshow(
      df.T,
      aspect = 'auto',
      labels = dict(x='Cell line', y='Gene', color=color_label)
  )

  fig.update_layout(
      title=f'Oncoprint of {name}'
  )
  return fig

# Keep genes mutated in at least 5% of cell lines
mutation_freq = mutation_pivot.sum(axis=0)
selected_genes = mutation_freq[mutation_freq >= 0.05 * len(mutation_pivot)].index

fig = plot_oncoheatmap_binary(mutation_pivot[selected_genes], 'Genes mutated in ≥5% of cell lines', 'Mutation')
fig.show()

## Copy Number Aberration Data exploration

In [ ]:
CNA_df.head()

,model_name,model_id,symbol,total_copy_number,cn_category,loh,focal,cancer_driver
45,M14,SIDM00003,HEPHL1,3.0,Neutral,0.0,0.0,0
46,M14,SIDM00003,GTF3C4,3.0,Neutral,0.0,0.0,0
47,M14,SIDM00003,GTF3C3,3.0,Neutral,0.0,0.0,0
48,M14,SIDM00003,GTF3C2,3.0,Neutral,0.0,0.0,0
49,M14,SIDM00003,GTF3C1,3.0,Neutral,0.0,0.0,0


In [ ]:
CNA_df['cn_category'].unique()

array(['Neutral', 'Loss', 'Gain', 'Amplification', 'Deletion'],
      dtype=object)

In [ ]:
CNA_df['cn_values'] = CNA_df['cn_category'].map({
    'Gain': 1,
    'Amplification': 1,
    'Loss': -1,
    'Deletion': -1,
    'Neutral': 0
})

CNA_pivot = CNA_df.pivot_table(
    index='model_id',
    columns='symbol',
    values='cn_values',
    aggfunc='max'
)

CNA_pivot.head()

symbol,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11B,ZYX,ZZEF1,ZZZ3
model_id,,,,,,,,,,,,,,,,,,,,,
SIDM00003,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,NaN,NaN,1.0,1.0,1.0,1.0,1.0
SIDM00023,-1.0,0.0,1.0,1.0,-1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,NaN,NaN,0.0,0.0,0.0,0.0,0.0
SIDM00040,0.0,0.0,-1.0,-1.0,-1.0,-1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,NaN,NaN,-1.0,0.0,0.0,-1.0,0.0
SIDM00041,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,1.0,NaN,NaN,1.0,0.0,0.0,0.0,0.0
SIDM00042,1.0,0.0,1.0,1.0,-1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,NaN,NaN,0.0,-1.0,0.0,0.0,-1.0


In [ ]:
print(f'Number of NaNs before data imputation: {CNA_pivot.isna().sum().sum()}')
print(f'Shape before data imputation: {CNA_pivot.shape}')

# Keep columns where proportion of NaN values is <= 10%
threshold = 0.1
gene_na = CNA_pivot.isna().mean(axis=0)
keep_cols = gene_na[gene_na <= threshold].index
CNA_clean = CNA_pivot[keep_cols]

# CNA values are on a scale where 0 = neutral copy number (no gain or loss)
# Treat remaining sporadic NaNs as neutral rather than drop, since these are isolated missing calls
CNA_clean = CNA_clean.fillna(0)

print(f'Number of NaNs after data imputation: {CNA_clean.isna().sum().sum()}')
print(f'Shape after data imputation: {CNA_clean.shape}')

Number of NaNs before data imputation: 437436
Shape before data imputation: (961, 18000)
Number of NaNs after data imputation: 0
Shape after data imputation: (961, 17256)


In [ ]:
def plot_oncoheatmap_diverging(df, name, color_label):
  '''
  Plots an oncoprint-style heatmap with a diverging blue-white-red colour scale.

  Args:
    df (pandas.DataFrame): Data of shape (cell_lines, genes).
    name (str): Label used to plot title.
    color_label (str): Label used for colour bar.

  Returns:
    fig (plotly.graph_objects.Figure): Heatmap with cell lines on x-axis, genes on y-axis.
  '''
  fig = px.imshow(
      df.T,
      color_continuous_scale=[
          [0.0, 'blue'],
          [0.5, 'white'],
          [1.0, 'red']
      ],
      aspect = 'auto',
      labels = dict(x='Cell line', y='Gene', color=color_label)
  )

  fig.update_layout(
      title=f'Oncoprint of {name}'
  )
  return fig

fig = plot_oncoheatmap(CNA_pivot, 'Gene Alteration Data', 'Gene Alteration')
fig.show()

In [ ]:
drug_df.to_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Clean_Dataframes/drug.parquet')
rnaseq_pivot.to_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Clean_Dataframes/rnaseq_clean.parquet')
mutation_pivot.to_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Clean_Dataframes/mutation_clean.parquet')
CNA_clean.to_parquet('/content/drive/MyDrive/Colab_Notebooks/Projects/Paper_to_code/Data/Clean_Dataframes/CNA_clean.parquet')